# ThaiFood AI 150 — Kaggle GPU

This notebook trains a **candidate** 150-class model and writes a checkpoint after every epoch. It **does not** update the public Hugging Face app. Before running, add a Kaggle input Dataset with `data/train`, `data/val`, `data/test`, `data/class_names.json`, plus the original `food_model_best.pth` (50-class). Kaggle Internet must be enabled to clone GitHub; otherwise upload the scripts as a Kaggle Dataset.

**Important:** Save a Kaggle notebook version with outputs at the end of a successful session, and attach the previous output as an Input to resume later. Files in `/kaggle/working` are not a permanent cross-session store. Dataset owners limit usage to non-commercial research/education.

In [ ]:
import os, subprocess, sys, shutil
import torch
from pathlib import Path
print("CUDA available:", torch.cuda.is_available())
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("nvidia-smi not found. Enable a GPU in Kaggle Notebook Settings > Accelerator.")
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: PyTorch cannot see CUDA. Training on CPU will be much slower.")
repo = Path("/kaggle/working/lab-ai-69")
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "thaifood-space-deploy", "https://github.com/nonthawat2548/lab-ai-69.git", str(repo)], check=True)
else:
    print("Repository already cloned:", repo)


## Set your input paths

In the right sidebar, **Add Input** for the prepared image dataset and original model. If you have a saved checkpoint from another Kaggle run, also attach its Notebook Output or a private Kaggle Dataset. Set the paths to the **actual mounted paths**, not the example placeholders.

In [ ]:
# EDIT these values to match the right sidebar under /kaggle/input
DATA_DIR = "/kaggle/input/YOUR-PREPARED-DATASET/data"
OLD_MODEL = "/kaggle/input/YOUR-LEGACY-MODEL/food_model_best.pth"
RESUME_CHECKPOINT = ""  # e.g. "/kaggle/input/YOUR-PREVIOUS-OUTPUT/last_checkpoint.pth"
OUTPUT_DIR = "/kaggle/working/thai_food_150"
TOTAL_EPOCHS = 20
BATCH_SIZE = 8

assert Path(DATA_DIR, "class_names.json").is_file(), "Missing 150-class prepared data; read the README first"
assert Path(OLD_MODEL).is_file(), "Missing original 50-class checkpoint"
if RESUME_CHECKPOINT:
    assert Path(RESUME_CHECKPOINT).is_file(), "Missing checkpoint to resume"


In [ ]:
import subprocess, sys
cmd = [sys.executable, "/kaggle/working/lab-ai-69/thai_food_150/kaggle_train.py",
       "--data", DATA_DIR, "--old-checkpoint", OLD_MODEL,
       "--output-dir", OUTPUT_DIR, "--epochs", str(TOTAL_EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--workers", "2"]
if RESUME_CHECKPOINT:
    cmd += ["--resume", RESUME_CHECKPOINT]
subprocess.run(cmd, check=True)


In [ ]:
from pathlib import Path
out = Path(OUTPUT_DIR)
print("Generated files:")
for f in sorted(out.glob("*")):
    print(f.name, f.stat().st_size, "bytes")
print("\nSave a Version with outputs; attach that Version's outputs next time and set RESUME_CHECKPOINT.")
